In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
# 1. Generate synthetic data
np.random.seed(42)
n_samples = 200
# Class 0: Younger, lower income
age_0 = np.random.randint(18, 35, size=n_samples // 2)
income_0 = np.random.randint(20, 50, size=n_samples // 2)
# Class 1: Older, higher income
age_1 = np.random.randint(30, 60, size=n_samples // 2)
income_1 = np.random.randint(60, 120, size=n_samples // 2)
X = np.vstack((np.column_stack((age_0, income_0)),
               np.column_stack((age_1, income_1))))
y = np.array([0] * (n_samples // 2) + [1] * (n_samples // 2))
# 2. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)
# 3. Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# 4. Hyperparameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # 1=Manhattan, 2=Euclidean
}
# 5. Grid Search with Cross Validation
knn = KNeighborsClassifier()
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)
# 6. Best model and prediction
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_test_scaled)
print("Best Parameters:", grid_search.best_params_)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.2f}")
# 7. Confusion matrix
ConfusionMatrixDisplay.from_estimator(best_knn, X_test_scaled, y_test)
plt.title("Confusion Matrix - Best KNN")
plt.grid(False)
plt.show()
# 8. Decision boundary plot
def plot_decision_boundary(X, y, model):
    h = 0.1
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k')
    plt.xlabel('Age (scaled)')
    plt.ylabel('Income (scaled)')
    plt.title("KNN Decision Boundary (Best Model)")
    plt.grid(True)
    plt.show()
plot_decision_boundary(X_test_scaled, y_test, best_knn)
'''Output -
Best Parameters: {'n_neighbors': 3, 'p': 1, 'weights': 'uniform'}
Test Accuracy: 1.00
'''
